In [1]:
!pip install torch transformers

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from sklearn.linear_model import SGDClassifier
import random
import string

class TCAV:
    """ Class for concept activation vectors for Keras models.

    Attributes:
        model: a roBerta LLM loaded trough huggingface API
        tokenizer: a tokenizer for roBerta
        cav: A numpy array containing the concept activation vector
        sensitivity: A numpy array containing sensitivities
        y_labels: A numpy array containing class labels
        bottleneck: a int indicating to which hidden layer extract the activations
    """

    def __init__(self, model=None, tokenizer=None):
        """ Inizializza la classe con variabili vuote """
        self.model = model
        self.tokenizer = tokenizer
        self.cav = None
        self.sensitivity = None
        self.y_labels = None
        self.bottleneck = None
        self.model_activations = {} #where to store activations
        
        return
    
    def hook_fn(self, name):
        if name not in self.model_activations.keys():
            self.model_activations[name] = []
        def fn(module, input, output):
            print("leaf",output.is_leaf)
            x = output
            # print("leaf",output[0].is_leaf)
            # x = output[0]
            x.requires_grad_(True)
            x.retain_grad()
            self.model_activations[name].append(x)
            print("extracted activations", output.shape)
        return fn

    def set_model(self, model):
        """ Set the model """
        self.model = model
        return

    def set_tokenizer(self, tokenizer):
        """ Set the tokenizer """
        self.tokenizer = tokenizer
        return

    def split_model(self, bottleneck):
        """ Set the hook at the bottleneck layer """
        if bottleneck < 0 or bottleneck >= len(list(self.model.children())):
            raise ValueError("Il layer di bottleneck deve essere valido!")
        
        layers = list(self.model.children())
        print(layers)
        self.bottleneck = str(bottleneck)   
        
        self.model.classifier.dense.register_forward_hook(self.hook_fn(str(bottleneck)))
        #self.model.roberta.encoder.layer[bottleneck].register_forward_hook(self.hook_fn(str(bottleneck)))
        
        return


    def _create_counterexamples(self, x_concept):
        """ Creates random counterexamples to a series of concept inputs """
        n = len(x_concept)
    
        counterexamples = []
        for i in range(n):
            l = len(x_concept[i])
            counterexamples.append(''.join(random.choices(string.printable, k=l)))
        return counterexamples

    def _tokenize(self, inputs):
        """ Tokenize the inputs (if tokenizer is provided) """
        print(len(inputs))
        if self.tokenizer is not None:
            x = self.tokenizer(inputs, return_tensors="pt", padding=True)
            print("tokenizer", x["input_ids"].shape)
            return x
        return inputs

    def train_cav(self, x_concept):
        """ Train and extract the Concept Activation Vector """
        
        counterexamples = self._create_counterexamples(x_concept)
        tmp = x_concept + counterexamples
        x_train_concept = self._tokenize(tmp)
        y_train_concept = torch.cat((torch.ones(len(x_concept)), torch.zeros(len(counterexamples))))
        
        print("calculating cavs")
        # Obtain activations of concept and counterexamples
        with torch.no_grad():
            _ = self.model(**x_train_concept)
            print("attentions dimensions:", self.model_activations[self.bottleneck][0])
            if(len(self.model_activations[self.bottleneck][0].shape)>2):
                concept_activations = self.model_activations[self.bottleneck][0].reshape(self.model_activations[self.bottleneck][0].shape[0],-1)
            else:
                concept_activations = self.model_activations[self.bottleneck][0]
            print("concept activations shape", concept_activations.shape)
        # print(concept_activations.shape)
        ### Concatenate token by token
        
        # Train linear classifier
        lm = SGDClassifier(loss="perceptron", eta0=1, learning_rate="constant", penalty=None)
        lm.fit(concept_activations.detach().numpy(), y_train_concept.numpy())
        self.cav = -lm.coef_.T
        print("cav", len(self.cav))
        
        self.model_activations[self.bottleneck] = [] #once calculated all results, reset for next operations                           
        return
        
    def calculate_sensitivity(self, x_train, y_train, device="cpu"):
        """
        Versione PyTorch della funzione, con commenti che rimandano
        ai passaggi originali in Keras.
        """
        
        print("calculating sensitivity")
        x_train = self._tokenize(x_train)
        # print(x_train)
        
        # Calculate the output and obtain activations
        x_train = x_train.to(device)
        output = self.model(**x_train)

        activations = self.model_activations[self.bottleneck][0].reshape(self.model_activations[self.bottleneck][0].shape[0],-1) # Prendi l'ultima attivazione del bottleneck
        print("output logits", output.logits.shape)
        print("activation shape", activations.shape)
        
        # Control the format
        if isinstance(y_train, list):
            y_train = np.array(y_train)
            # print(y_train)
        if not isinstance(y_train, torch.Tensor):
            y_train = torch.from_numpy(y_train)
        y_labels = y_train.view(-1).to(device)
        # print(y_labels)

        # Make sure you can calculate the gradient on the activations
        activations.requires_grad_(True)

        # Define and compute the loss
        loss = F.cross_entropy(output.logits, y_labels)
        print("loss", loss)
        print("activations",activations.is_leaf, activations)
        print("activations requires grad", activations.requires_grad)

        # Calculate the gradient
        loss.backward()
        
        grads = activations.grad
        # grads = torch.autograd.grad(loss, activations, allow_unused=True)
        print("grads", grads)

        # Scalar product
        cav_tensor = self.cav
        sensitivity = np.dot(grads, cav_tensor)

        # Saving sensitivity
        self.sensitivity = sensitivity.detach().cpu().numpy()
        self.y_labels    = y_train.detach().cpu().numpy().reshape(-1)

        return
        
    def print_sensitivity(self):
        """ Print sensitivity in a readable way """
        if isinstance(self.y_labels, list):
            self.y_labels = np.array(self.y_labels)

        num_labels = self.y_labels.shape[1] if len(self.y_labels.shape) > 1 else 1

        for label_idx in range(num_labels):
            value = np.sum(self.sensitivity[np.where(self.y_labels == label_idx)[0]] > 0) / np.where(self.y_labels == label_idx)[0].shape[0]
            print(f"Sensitivity for label {label_idx} is: {x}")
            
        return

/home/students/fmarmello/.local/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/students/fmarmello/.local/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
import json
import os
import torch
CLASSES = classes = {'go': 0, 'java': 1, 'javascript': 2, 'php': 3, 'python': 4, 'ruby': 5}
CONCEPTS= ["php_function_declarations","comments", "function_declarations", "python_function_declarations", "go_function_declarations", "java_function_declarations", "javascript_function_declarations", "ruby_function_declarations"]
CLASSES_TO_EXAMINE = ['go', 'java', 'javascript', 'php', 'python', 'ruby']
MODEL_NAME = "huggingface/CodeBERTa-language-id"

In [4]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(CLASSES_TO_EXAMINE))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Some weights of the model checkpoint at huggingface/CodeBERTa-language-id were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [5]:
for concept in CONCEPTS:
    data = []
    with open(f"./data/code_classification/{concept}_dataset.jsonl", "r") as f:
        concept_examples = [json.loads(line) for line in f]
        
        for x in concept_examples:
            data.append({'text': x['text'], 'label': CLASSES[x['language']]})

    for c in CLASSES_TO_EXAMINE:
        # calculate TCAVs
        print(f"Calculating TCAVs for {concept} concept, class {c}")
        tcav_object = TCAV()
        tcav_object.set_model(model)
        tcav_object.set_tokenizer(tokenizer)
        tcav_object.split_model(1)
        tcav_object.train_cav([d["text"] for d in data])
        tcav_object.calculate_sensitivity([d["text"] for d in data], [d["label"] for d in data])
        tcav_object.print_sensitivity()

Calculating TCAVs for php_function_declarations concept, class go
[RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(52000, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-5): 6 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): Laye

/tmp/ipykernel_10495/4192295307.py:161: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  grads = activations.grad


TypeError: unsupported operand type(s) for *: 'NoneType' and 'float'